# Blockchain Agriculture — Case Study

> How to justify blockchain for farming? Put numbers on the table.

---

### The business question

A regional farm business ("Eifel42 Agra Corp.") is deciding between three blockchain features. The exact values come from the scenario configuration below and from `config/blockchain.yaml`.

| Hypothesis | Feature | What it aims to improve |
|:--:|---|---|
| **H1** | Simplified user interface | Retention and conversion through lower UX friction |
| **H2** | Complete traceability | New business value and trust through provenance |
| **H3** | Automatic expiry alerts | Waste reduction and operational efficiency |

### Project config (single source of truth)
- Config file: `apps/fhs/notebooks/config/blockchain.yaml`
- Validated by `scenario_schema.json`
- All notebooks load this file; change it once, rerun from the top.
- `annual_growth_rate` controls how business value grows each year.
- The tool simulates all 3 years with the configured scenario count per year.

*Previous: [01-getting-started.ipynb](01-getting-started.ipynb) · Next: [T01 — Distribution Guide](tutorial/01-distribution-guide.ipynb)*


Regulatory features are built to meet legal or compliance requirements and are prioritized regardless of immediate financial return.


**Key takeaway** — Replace opinions with a simple risk/return view that shows what each feature delivers and how low the floor can go.

In [ ]:
from fhs.application import BlockchainCaseStudyService
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import (
    COLORS,
    palette,
    show,
)
from fhs.presentation.notebook.charts import (
    plot_portfolio_distribution,
    plot_risk_comparison,
)

setup = notebook_setup("blockchain", editable=True)
scenario = setup.scenario

if scenario is None:
    raise RuntimeError("Scenario setup could not be initialized")
feature_keys = sorted(scenario.features_by_key)
case = BlockchainCaseStudyService(seed=scenario.seed, scenarios=scenario.scenarios)

year1 = case.simulate_year1(scenario.features_by_key)
chart_data = case.chart_payload(year1)

icons = {"H1": "📱", "H2": "🔗", "H3": "⚠️"}
colors = {"H1": COLORS.primary, "H2": COLORS.secondary, "H3": COLORS.accent}

show.info(
    f"✅ Scenario loaded: {len(feature_keys)} features, "
    f"{case.scenarios:,} simulations per feature"
)

In [ ]:
show.info(
    "✅ Features loaded: " + ", ".join(year1[key].feature.name for key in feature_keys)
)

## Feature details (live from config)

Everything below is pulled directly from `config/blockchain.yaml`. It shows what a product owner or risk manager needs at a glance: users, conversion model, value per conversion, uncertainty, delivery risk, cost, dependency, and release.


In [ ]:
show.feature_overview(
    [year1[key].feature for key in feature_keys],
    scenario.strategy,
    config_path=scenario.config_path,
    colors=colors,
    expected_business_value_func=case.expected_business_value,
    standalone_roi_func=case.roi,
)

### Background — Year-1 delivery risk

**Year-1 delivery risk** is a simple number:

$$\text{Year-1 Delivery Risk} = \text{Expected BV} \times p_{\text{non-delivery}}$$

- **Product Owner:** How much business value is at risk if this feature is not delivered?
- **Risk Manager:** Expected business value exposure from delivery failure in Year 1.
- This is **not** the same as the Monte Carlo **VaR 95% floor** shown later.
- Use it as planning context for prioritization and delivery risk discussions.

The feature overview above shows each feature's delivery risk value.

### Team dependencies (from config)

Dependencies are loaded from `dependency_cluster` in the same scenario file.

- Shared team/dependency means features can fail together.
- This notebook keeps calculations independent for readability.
- For correlated portfolio risk, see [A01 — Portfolio Advisor](advanced/01-portfolio-advisor.ipynb) and [A02 — Portfolio Risk Dashboard](advanced/02-portfolio-risk-dashboard.ipynb).

---

## How the simulation works — step by step

We walk through the calculation using **H1** as an example.

### Step 1: The deterministic formula

Without uncertainty the tool computes one number:

$$\text{Expected Business Value} = \text{Users} \times \text{Conversion Rate} \times \text{Value per Conversion}$$

### Step 2: Add uncertainty

The conversion rate is **not fixed**. The tool draws a slightly different rate for every scenario.

- **Uncertainty below 30 %** → symmetric bell curve (normal distribution)
- **Uncertainty 30 % or higher** → right-skewed curve (lognormal distribution)

The code cell below shows which distribution H1 uses based on its current config.

### Step 3: Acceptance model

When a feature uses the **binomial** acceptance model, each user independently converts (or not). This produces realistic whole-number conversion counts instead of a single averaged percentage.

### Step 4: Run the configured number of scenarios

Every scenario gives a different business value result. Collect them all and you get a full picture: expected value, downside floor (VaR), and upside ceiling.


In [ ]:
# ── H1 Example Calculation ──────────────────────────────────────────
h1 = year1["H1"].feature
h1_summary = year1["H1"]

det_business_value = case.expected_business_value(h1)
dist_name = "lognormal" if h1.uncertainty >= 0.3 else "normal"

show.formula(
    "Step 1 — Deterministic calculation",
    f"{h1.expected_users:,} users × {h1.conversion_rate:.0%} conversion × €{h1.business_value_per_conversion:,.2f}/conversion",
    result=f"€{det_business_value:,.0f}",
)

In [ ]:
sample_scenarios = min(1_000, scenario.scenarios)
sample_rates = case.sample_conversion_rates(h1, scenarios=sample_scenarios)

In [ ]:
show.samples(
    headers=("Scenario", "Rate", "Conversions", "Business Value"),
    rows=[
        (
            f"Scenario {idx}",
            f"{rate:.3f}",
            f"{int(rate * h1.expected_users):,}",
            f"€{int(rate * h1.expected_users) * h1.business_value_per_conversion:,.0f}",
        )
        for idx, rate in enumerate(sample_rates[:5], 1)
    ],
    title="Step 2 — Uncertainty creates different rates",
    description=(
        f"Base rate: {h1.conversion_rate:.0%}, "
        f"Uncertainty: ±{h1.uncertainty:.0%} "
        f"→ auto-selects {dist_name} distribution"
    ),
    footer="5 of 1,000 sample draws shown",
)

In [ ]:
show.metrics(
    [
        ("Expected (mean)", f"€{h1_summary.expected_eur:,.0f}", None),
        ("VaR 95 % (floor)", f"€{h1_summary.var_95_eur:,.0f}", COLORS.danger),
        ("P95 (ceiling)", f"€{h1_summary.p95_eur:,.0f}", COLORS.secondary),
        ("Std Dev (spread)", f"€{h1_summary.std_eur:,.0f}", None),
    ],
    title=f"Step 3 — Full simulation: {case.scenarios:,} scenarios",
)

**What just happened**

1. **Step 1** gave us one number — the "spreadsheet answer". No uncertainty, no range.
2. **Step 2** drew five sample conversion rates from the configured distribution. Some came out above average, some below — that is the uncertainty at work.
3. **Step 3** ran the full Monte Carlo with the configured scenario count. Now we can read off the expected value, the downside floor (VaR 95 %), and the upside ceiling (P95).

> **Key insight:** The deterministic number and the simulation mean are close — but the simulation also tells you the *floor* and the *spread*. That is the information a simple spreadsheet does not give you.

Next we run all three features and compare them.


---

## 1) How much business value can each feature deliver?

Nobody knows exactly how many customers will adopt a new feature. So instead of guessing one number, we let the computer **try the configured number of different scenarios** — some optimistic, some pessimistic, most somewhere in between.

The result is a chart that shows **all possible business value outcomes** and how likely each one is. Four numbers matter most:

| Number | What it tells you |
|---|---|
| **Expected** | The average outcome — use this for planning |
| **VaR 95% (floor)** | 95 out of 100 scenarios beat this number — use this for worst-case budgets |
| **CVaR 95% (tail)** | If you land in the worst 5 %, this is the average you still make — the deepest downside |
| **P95 (ceiling)** | Only 5 out of 100 scenarios reach this high — do not plan with this |


In [ ]:
cards = []
for key in feature_keys:
    expected = float(year1[key].expected_eur)
    spread = float(year1[key].std_eur) / expected if expected else 0.0
    cards.append(
        show.feature_risk_card(
            year1[key].feature.name,
            expected,
            float(year1[key].var_95_eur),
            spread,
            cvar95_eur=float(year1[key].cvar_95_eur),
            icon=icons.get(key, "📊"),
            color=colors.get(key, COLORS.primary),
        )
    )

show.scorecard(cards, title="All Three Hypotheses")

### Year 1 at a glance — business value vs. costs

The table below puts all Year-1 numbers in one place. Each row is one feature; the last row is the combined portfolio.

| Column group | What it shows |
|---|---|
| **a) Business Value** | Expected, VaR 95% floor, CVaR 95% tail — from Monte Carlo |
| **b) Variant 1 — Full Investment** | Full development cost + year-1 opex deducted; net value in year 1 |
| **c) Variant 2 — Year-1 P&L** | Annual installment charge + year-1 opex deducted; net value in year 1 |

A **green +** means the feature covers its costs in year 1. A **red −** means it does not — which may still be fine over a multi-year horizon (see Notebook 03).

In [ ]:
portfolio_overview = case.portfolio_year1(year1)

show.year1_overview(
    case.year1_overview_rows(year1, portfolio_overview),
    title="Year 1 Overview — Business Value vs. Costs",
)

**Reading the table above**

Each row is one feature. The last row aggregates all three into the portfolio total.

- **Expected** — the mean outcome across all simulated scenarios. Use this for planning.
- **VaR 95% floor** — 95 out of 100 scenarios produced a business value *above* this number.
- **CVaR 95% tail** — if you land in the worst 5 %, this is the average you still make.
- **Net Year 1 (V1)** — Expected minus the full development investment and one year of operating cost.
- **Net Year 1 (V2)** — Expected minus the annual installment charge and one year of operating cost.

> Portfolio VaR and CVaR come from the **combined Monte Carlo**, not a simple sum — so they account for diversification.

#### Variant 2 — Proportional installment view

The overview table above shows two cost options in the **Net Year 1** columns.
Variant 2 replaces the full one-time investment charge with the **annual installment amount** — the investment spread evenly over the asset lifetime:

$$
\text{Net}_{\text{V2}} = BV - \frac{C_{\text{dev}}}{n_{\text{years}}} - C_{\text{opex,\,Y1}}
$$

This is the **P\&L / accounting perspective**: the income statement sees only the installment slice each year, not the full cash outlay on day one.
The table below shows how the portfolio looks under Variant 2 across all four risk scenarios.

In [ ]:
show.portfolio_pl_variants(
    case.portfolio_pl_variants(year1, portfolio_overview),
    title="Portfolio Year-1 P&L — Installment View (Variant 2)",
)

### Compare the downside across features

Now we overlay all three features on the same chart so you can compare them directly: which one has the highest business value, and which one carries the most risk?


In [ ]:
# ── Risk comparison: overlay + ranking ────────────────────────────────
plot_risk_comparison(
    chart_data,
    colors={
        "H1": palette["primary"],
        "H2": palette["secondary"],
        "H3": palette["accent"],
    },
    title="Blockchain Hypotheses — H1, H2, H3",
)

show.risk_summary_table(
    rows=[
        (
            year1[key].feature.name,
            f"€{year1[key].expected_eur:,.0f}",
            f"€{year1[key].var_95_eur:,.0f}",
        )
        for key in feature_keys
    ],
    title="Risk Summary",
)

**Reading the charts above**

**Left chart (overlapping histograms):** All three features share the same x-axis so you can compare their spread at a glance. Features with higher uncertainty produce wider distributions. Features with uncertainty ≥ 30 % appear right-skewed; those below 30 % look more symmetric.

**Right chart (bar comparison):** The solid bar is the expected business value, the faded bar is the VaR 95 % floor. The gap between them is your **downside exposure** — the business value you could lose compared to plan in a bad scenario.

**What to ask yourself:** "Is this gap acceptable, or should we invest in user research to reduce uncertainty?" Lowering a feature's uncertainty shrinks the gap and makes the outcome more predictable (see Tutorial 01 for details).

Read the dynamic info box directly above for the exact numbers under the current configuration.


### How bad can it really get? (CVaR)

VaR 95 % tells you where the floor is. But what happens *below* that floor?

**CVaR** (Conditional Value at Risk) answers: "If we end up in the worst 5 % of scenarios, what business value do we still make on average?"

A quick example:
- VaR 95 % = €200k and CVaR = €190k → the bad scenarios stay close to the floor. **Low tail risk.**
- VaR 95 % = €200k and CVaR = €50k → some bad scenarios are much worse. **High tail risk.**

The smaller the gap between VaR and CVaR, the more predictable your worst case.


In [ ]:
show.comparison(
    [
        (
            year1[key].feature.name,
            float(year1[key].expected_eur),
            float(year1[key].var_95_eur),
            float(year1[key].cvar_95_eur),
        )
        for key in feature_keys
    ],
    title="Downside Exposure Comparison",
)

**Reading the table above**

Compare the VaR 95% column with the CVaR column for each feature.

- A **small gap** means the bad scenarios stay close to the floor.
- A **large gap** means the worst tail contains more severe downside than VaR alone suggests.

This is the quickest way to see whether a feature has a stable downside profile or hidden tail risk.


---

## 2) What if we build all three?

So far we looked at each feature alone. Now we combine all three to see the **total business value picture**. We simply add up the configured business value scenarios from H1, H2, and H3.


In [ ]:
# ── Portfolio distribution ────────────────────────────────────────────
portfolio = case.portfolio_year1(year1)
portfolio_scenarios = sum(year1[key].scenarios_eur for key in feature_keys)
portfolio_contributions = {
    year1[key].feature.name: year1[key].expected_eur for key in feature_keys
}

plot_portfolio_distribution(
    portfolio_scenarios,
    portfolio_contributions,
    cvar_95=portfolio.cvar_95,
)

show.portfolio(
    portfolio.expected,
    portfolio.var_95,
    portfolio.std_dev,
    cvar95=portfolio.cvar_95,
)

**Reading the charts above**

**Left chart:** The combined portfolio distribution. If it looks smoother than the single-feature charts, that is the diversification effect.

**Right chart:** The contribution breakdown by feature. This shows which feature dominates the portfolio and which ones mainly improve resilience or add incremental value.

Read the portfolio KPI card directly above for the current expected value, downside floor, and volatility under the active configuration.


### How do combined costs affect the portfolio bottom line?

The charts above show what the portfolio delivers. Now we subtract costs:

| View | What is deducted | Use it when |
|---|---|---|
| **Variant 1 (cash-flow)** | Full development investment + year-1 opex | Comparing against annual budget ceiling |
| **Variant 2 (P&L / accounting)** | Year-1 installment charge + year-1 opex | Finance team capitalises development spend |

Four business value statistics are shown so you can see the picture at every point of the distribution — not just the expected value.

In [ ]:
show.portfolio_pl_variants(
    case.portfolio_pl_variants(year1, portfolio),
    title="Portfolio Year-1 P&L — Full Investment vs. Installment",
)

---

## 2b) What does it cost to run each feature every year?

Building a feature is a one-time investment. **Running it in production costs money every year** — servers, cloud services, third-party APIs, support.

We simulate those annual operating costs with a shared inflation factor drawn from a uniform distribution between 0 % and the configured maximum (default: 25 %). One factor is drawn per scenario and applied to all features equally — the assumption is that infrastructure cost drivers (hosting, energy, licences) move together.

| Column | What it tells you |
|---|---|
| **Base (EUR/yr)** | Configured annual operating cost with no inflation |
| **Expected** | Average simulated cost across all scenarios |
| **Worst Case** | Cost at the maximum inflation rate |
| **Inflation Impact** | Extra spend compared to the base (Worst Case − Base) |

In [ ]:
op_costs = case.simulate_operating_costs(
    scenario.features_by_key,
    cost_inflation_max=scenario.cost_inflation_max,
)
show.operating_costs(
    op_costs, title="Annual Operating Cost Simulation — shared inflation up to 25%"
)

---

## 3) Recommendation for the board

### Question 01 — Initiative Budget Check

**Decision answer first (beginner view):**
- Product Owner: Can we fund **H1 + H2 + H3** now?
- Risk Manager: How much **budget buffer** is left after funding all three?

The next output answers this first in one compact table (all three features side by side), then gives the portfolio recommendation.


In [ ]:
# ── Portfolio Ranking & GO/NO-GO Decision ───────────────────────────
from fhs.application import BoardRecommendationService

selected_features = [year1[key].feature for key in feature_keys]

show.budget_status(
    scenario.budget,
    selected_features,
    source=scenario.config_source,
    scenarios=case.scenarios,
    title="Question 01 — Initiative Budget Check (all three features)",
)

### Question 02 — In which order should we roll out the features?

**Decision answer first:**
- Product Owner: Which feature should go live first to maximise early business value return?
- Risk Manager: Does the expected portfolio business value justify a GO recommendation?

The KPI cards below rank all three features by simulated expected business value — highest first = Phase 1.
The board callout gives the GO / Conditional GO / Review recommendation from the same model.

In [ ]:
rec_service = BoardRecommendationService(seed=case.seed, scenarios=case.scenarios)

In [ ]:
recommendation = rec_service.generate_recommendation(
    selected_features,
    scenario.biz_values,
    discount_rate=scenario.discount_rate,
    year1_results=year1,
)

phase_colors = [COLORS.secondary, COLORS.secondary, COLORS.accent]
show.kpi(
    *(
        show.kpi_card(value, label, color=phase_colors[min(i, len(phase_colors) - 1)])
        for i, (value, label) in enumerate(recommendation.kpi_phase_data)
    )
)

In [ ]:
show.metrics(recommendation.portfolio_kpi_rows, title="Portfolio — combined view")

if recommendation.decision_level == "success":
    show.success(recommendation.decision_message)
else:
    show.warning(recommendation.decision_message)

show.note(
    f"Based on {case.scenarios:,} Monte Carlo scenarios per hypothesis", compact=True
)

### Background — How the budget check works

The budget check uses a simple formula:

$$C_{\text{total}} = \sum C_{\text{feature}}$$

$$B_{\text{remaining}} = B_{\text{budget}} - C_{\text{total}}$$

Decision rule:

$$B_{\text{remaining}} \ge 0 \Rightarrow \text{all features fit the budget}$$

$$B_{\text{remaining}} < 0 \Rightarrow \text{portfolio is over budget}$$

All values come from `config/blockchain.yaml`. Rerunning the notebook keeps the answer up to date.

---

**Want the full financial view (ROI, NPV, IRR)?**

Go to **[03 — Capital Budgeting (ROI, NPV, IRR)](03-blockchain-case-study-capital-budgeting.ipynb)**.

---

## Summary

| What you learned | Key takeaway |
| --- | --- |
| Three hypotheses quantified | Expected value, downside floor (VaR 95 %), and CVaR for each feature |
| Downside exposure measured | CVaR shows the average of the worst 5 % of scenarios |
| Portfolio view | Total value and floor when all features are combined |
| Phased rollout | Features ranked by simulated expected business value (see KPI cards above) |
| Business case | Monte Carlo provides a clear, data-driven case for the board |

**Next:** Want to evaluate the financial return of these features? See [03 — Capital Budgeting (ROI, NPV, IRR)](03-blockchain-case-study-capital-budgeting.ipynb) to learn how CFOs decide which projects get funded.

---

## Notebook Navigation

| # | Notebook | What you learn |
|:-:|----------|----------------|
| 01 | [Getting Started](01-getting-started.ipynb) | One feature, 10,000 scenarios, VaR dashboard |
| 02 | **Blockchain Case Study** | ← You are here |
| 03 | [Capital Budgeting (ROI, NPV, IRR)](03-blockchain-case-study-capital-budgeting.ipynb) | Financial metrics for investment decisions |
| T01 | [Distribution Guide](tutorial/01-distribution-guide.ipynb) | Normal vs. Lognormal vs. Beta — pick the right model |
| 07 | [Executive Decision](07-blockchain-case-study-decision.ipynb) | All dimensions in one view, combined recommendation |
| A01 | [Portfolio Advisor](advanced/01-portfolio-advisor.ipynb) | Feature ranking, solver comparison (ILP/Exact/Greedy), runtime scaling, HHI |
| A02 | [Portfolio Risk Dashboard](advanced/02-portfolio-risk-dashboard.ipynb) | Risk layers L1/L2/L3, LLP impact, stress scenarios, budget risk path |

---

## Glossary

Use the central glossary notebook for all term definitions:

- [GLOSSARY.ipynb](GLOSSARY.ipynb)


---
**Previous:** [NB 01: Getting Started](01-getting-started.ipynb) | **Next:** [NB 03: Capital Budgeting](03-blockchain-case-study-capital-budgeting.ipynb)
